In [2]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import sys
from typing import Dict, List

import pandas as pd

from pathlib import Path

try:
    ROOT = Path(__file__).resolve().parents[1]
except NameError:
    ROOT = Path.cwd().parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


In [3]:
RAW_ROOT = ROOT / 'data' / 'raw' 
START_TIMES_PATH = ROOT / 'data' / 'raw' / 'session_start_times.csv'
PROCESSED_ROOT = ROOT / 'data' / 'processed' / 'aggregated_by_signal'
WINDOW_MINUTES = 60

CONDITION_MAP = {
    'no_wind': 'no_wind',
    'wind_low': 'wind_low',
    'wind_high': 'wind_high',
}

SIGNAL_GROUPS: Dict[str, List[str]] = {
    'T': [
        'T NECK',
        'T CHEST',
        'T ABDOMEN',
        'T DORSAL',
        'T HAND',
        'T SHIN',
        'T MICROCLIMATEFRONT',
        'T MICROCLIMATEBACK',
        'T ENVIRONMENT',
    ],
    'RH': [
        'RH NECK',
        'RH CHEST',
        'RH ABDOMEN',
        'RH DORSAL',
        'RH HAND',
        'RH SHIN',
        'RH MICROCLIMATEFRONT',
        'RH MICROCLIMATEBACK',
        'RH ENVIRONMENT',
    ],
    'HR': ['HR'],
}

OUTPUT_COLUMNS = [
    'subject_id',
    'condition',
    'signal_group',
    'signal_name',
    'timestamp',
    'elapsed_seconds',
    'elapsed_minutes',
    'value',
]

In [4]:
@dataclass(frozen=True)
class SessionInfo:
    subject_id: str
    condition: str
    start_time: pd.Timestamp

def normalize_condition(condition: str) -> str:
    key = condition.strip().lower()
    if key not in CONDITION_MAP:
        raise ValueError(
            f'Unsupported condition "{condition}". Allowed: {list(CONDITION_MAP)}'
        )
    return CONDITION_MAP[key]

def load_start_times(path: Path) -> List[SessionInfo]:
    if not path.exists():
        raise FileNotFoundError(
            f'Missing start-times file: {path}\n'
            'Create data/raw/session_start_times.csv with columns: '
            'subject_id,condition,start_time'
        )

    df = pd.read_csv(path)
    required = {'subject_id', 'condition', 'start_time'}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f'session_start_times.csv missing columns: {sorted(missing)}')

    df = df.copy()
    df['subject_id'] = df['subject_id'].astype(str).str.strip()
    df['condition'] = df['condition'].astype(str).str.strip()
    df['start_time'] = parse_datetime_series(df['start_time'])

    if df['start_time'].isna().any():
        bad = df.loc[df['start_time'].isna(), ['subject_id', 'condition', 'start_time']]
        raise ValueError(
            'Some start_time values could not be parsed. Accepted examples: '\
            '06/05/25 10:25:51 or 06/05/2025 10:25:51.\n'
            f'Problematic rows:\n{bad.to_string(index=False)}'
        )

    infos: List[SessionInfo] = []
    for row in df.itertuples(index=False):
        condition = normalize_condition(row.condition)
        infos.append(SessionInfo(row.subject_id, condition, row.start_time))
    return infos


def parse_datetime_series(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip()

    # normalizza l'orario: 9.24.36 -> 9:24:36
    # ma lascia intatta la data 23/04/26
    s = s.str.replace(
        r'(\d{1,2})\.(\d{2})\.(\d{2})$',
        r'\1:\2:\3',
        regex=True
    )

    formats = [
        '%d/%m/%y %H:%M:%S',
        '%d/%m/%Y %H:%M:%S',
        '%d/%m/%y %H:%M',
        '%d/%m/%Y %H:%M',
    ]

    best = None
    best_count = -1

    for fmt in formats:
        parsed = pd.to_datetime(s, format=fmt, errors='coerce')
        count = parsed.notna().sum()
        if count > best_count:
            best = parsed
            best_count = count

    if best is not None and best_count > 0:
        return best

    return pd.to_datetime(s, errors='coerce', dayfirst=True)

In [5]:
from dataclasses import dataclass
from pathlib import Path
from typing import List
import pandas as pd

# =========================
# PATHS / GLOBAL CONFIG
# =========================

ROOT = Path("/home/eleonorab/repository/convective_cooling")
RAW_ROOT = ROOT / "data" / "raw"
PROCESSED_ROOT = ROOT / "data" / "processed" / "aggregated_by_signal"
START_TIMES_PATH = RAW_ROOT / "session_start_times.csv"
WINDOW_MINUTES = 60

# =========================
# DATA STRUCTURE
# =========================

@dataclass
class SessionInfo:
    subject_id: str
    condition: str
    start_time: pd.Timestamp


# =========================
# GENERIC HELPERS
# =========================

def save_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def normalize_text(text: str) -> str:
    return ''.join(ch for ch in str(text).lower() if ch.isalnum())


def signal_output_name(signal_name: str) -> str:
    return signal_name.lower().replace(' ', '_') + '.csv'


def parse_datetime_series(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip()

    # converte orari tipo 9.24.36 in 9:24:36
    s = s.str.replace(
        r'(\d{1,2})\.(\d{2})\.(\d{2})$',
        r'\1:\2:\3',
        regex=True
    )

    formats = [
        '%d/%m/%y %H:%M:%S',
        '%d/%m/%Y %H:%M:%S',
        '%d/%m/%y %H:%M',
        '%d/%m/%Y %H:%M',
    ]

    best = None
    best_count = -1

    for fmt in formats:
        parsed = pd.to_datetime(s, format=fmt, errors='coerce')
        count = parsed.notna().sum()
        if count > best_count:
            best = parsed
            best_count = count

    if best is not None and best_count > 0:
        return best

    return pd.to_datetime(s, errors='coerce', dayfirst=True)


def load_start_times(path: Path) -> List[SessionInfo]:
    df = pd.read_csv(path, dtype=str)

    required_cols = {'subject_id', 'condition', 'start_time'}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f'Missing columns in start times file: {missing}')

    df['subject_id'] = df['subject_id'].astype(str).str.strip()
    df['condition'] = df['condition'].astype(str).str.strip()
    df['start_time'] = parse_datetime_series(df['start_time'])

    if df['start_time'].isna().any():
        bad_rows = df[df['start_time'].isna()]
        raise ValueError(
            'Some start_time values could not be parsed:\n'
            f'{bad_rows[["subject_id", "condition", "start_time"]]}'
        )

    return [
        SessionInfo(
            subject_id=row.subject_id,
            condition=row.condition,
            start_time=row.start_time,
        )
        for row in df.itertuples(index=False)
    ]

# =========================
# FILE DISCOVERY
# =========================

def find_signal_file(session_dir: Path, signal_group: str, expected_name: str) -> Path:
    folder = session_dir / signal_group
    if not folder.exists():
        raise FileNotFoundError(f'Missing folder: {folder}')

    expected_csv = folder / f'{expected_name}.csv'
    if expected_csv.exists():
        return expected_csv

    normalized_expected = normalize_text(expected_name)
    candidates = list(folder.glob('*.csv'))

    for path in candidates:
        if normalize_text(path.stem) == normalized_expected:
            return path

    raise FileNotFoundError(
        f'File for signal "{expected_name}" not found in {folder}. '
        f'Available csv: {[p.name for p in candidates]}'
    )

# =========================
# READERS
# =========================

def read_temperature_humidity_csv(path: Path) -> pd.DataFrame:
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    header_row = None
    for i, line in enumerate(lines):
        if 'Date/Time' in line:
            header_row = i
            break

    if header_row is None:
        raise ValueError(f'Could not find the header row containing "Date/Time" in {path}')

    data = pd.read_csv(
        path,
        skiprows=header_row,
        dtype=str,
        engine='python',
        sep=None
    )

    if data.empty:
        raise ValueError(f'{path} has no data rows after header.')

    # nei tuoi file il timestamp vero finisce nell'index
    data = data.reset_index()

    if data.shape[1] < 4:
        raise ValueError(f'Unexpected format in {path}. Columns found: {list(data.columns)}')

    data = data.iloc[:, :4].copy()
    data.columns = ['timestamp', 'unit', 'value_int', 'value_dec']

    data['timestamp'] = data['timestamp'].astype(str).str.strip()
    data['value_int'] = data['value_int'].astype(str).str.strip()
    data['value_dec'] = data['value_dec'].astype(str).str.strip()

    data['value'] = data['value_int'] + '.' + data['value_dec']

    data['timestamp'] = parse_datetime_series(data['timestamp'])
    data['value'] = pd.to_numeric(data['value'], errors='coerce')

    data = data.dropna(subset=['timestamp', 'value']).sort_values('timestamp').reset_index(drop=True)

    return data[['timestamp', 'value']]


def read_hr_csv(path: Path) -> pd.DataFrame:
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    header_row = None
    for i, line in enumerate(lines):
        line_low = line.lower()
        if 'time [utc-ofs=' in line_low and 'hr' in line_low:
            header_row = i
            break

    if header_row is None:
        raise ValueError(f'Could not find HR header row in {path}')

    data = pd.read_csv(
        path,
        skiprows=header_row,
        dtype=str,
        engine='python',
        sep=r'\t+|\s{2,}'
    )

    if data.empty:
        raise ValueError(f'{path} has no data rows after HR header.')

    data.columns = [str(c).strip().lower() for c in data.columns]

    time_col = None
    hr_col = None

    for c in data.columns:
        if c.startswith('time ') or c == 'time' or 'time [utc-ofs=' in c:
            time_col = c
        if c == 'hr' or c.endswith(' hr') or c.strip() == 'hr':
            hr_col = c

    if time_col is None or hr_col is None:
        raise ValueError(
            f'Could not identify HR columns in {path}. '
            f'Columns found: {list(data.columns)}'
        )

    data = data[[time_col, hr_col]].copy()
    data.columns = ['time_only', 'value']

    data['time_only'] = data['time_only'].astype(str).str.strip()
    data['value'] = data['value'].astype(str).str.replace(',', '.', regex=False).str.strip()
    data['value'] = pd.to_numeric(data['value'], errors='coerce')

    data = data.dropna(subset=['value']).reset_index(drop=True)

    return data[['time_only', 'value']]


def read_signal_csv(path: Path, signal_group: str) -> pd.DataFrame:
    if signal_group.upper() == 'HR':
        return read_hr_csv(path)
    return read_temperature_humidity_csv(path)

# =========================
# WINDOW EXTRACTION
# =========================

def slice_session_window(
    df: pd.DataFrame,
    start_time: pd.Timestamp,
    duration_minutes: int,
    signal_group: str
) -> pd.DataFrame:
    end_time = start_time + pd.Timedelta(minutes=duration_minutes)

    if signal_group.upper() in ['T', 'RH']:
        mask = (df['timestamp'] >= start_time) & (df['timestamp'] < end_time)
        window = df.loc[mask].copy()

        if window.empty:
            return window

        window['elapsed_seconds'] = (window['timestamp'] - start_time).dt.total_seconds().astype(int)
        window['elapsed_minutes'] = window['elapsed_seconds'] / 60.0
        return window

    if signal_group.upper() == 'HR':
        df = df.copy()

        parsed_times = pd.to_datetime(df['time_only'], format='%M:%S.%f', errors='coerce')
        if parsed_times.isna().all():
            parsed_times = pd.to_datetime(df['time_only'], format='%M:%S', errors='coerce')

        if parsed_times.isna().all():
            raise ValueError(f'Could not parse HR time column near session start {start_time}')

        base_date = start_time.normalize()

        df['timestamp'] = [
            base_date + pd.Timedelta(
                hours=start_time.hour,
                minutes=t.minute,
                seconds=t.second,
                microseconds=t.microsecond
            ) if pd.notna(t) else pd.NaT
            for t in parsed_times
        ]

        if not df.empty:
            while df['timestamp'].max() < start_time:
                df['timestamp'] = df['timestamp'] + pd.Timedelta(hours=1)

        mask = (df['timestamp'] >= start_time) & (df['timestamp'] < end_time)
        window = df.loc[mask].copy()

        if window.empty:
            return window

        window['elapsed_seconds'] = (window['timestamp'] - start_time).dt.total_seconds().astype(int)
        window['elapsed_minutes'] = window['elapsed_seconds'] / 60.0

        return window[['timestamp', 'value', 'elapsed_seconds', 'elapsed_minutes']]

    raise ValueError(f'Unsupported signal_group: {signal_group}')

session_infos = load_start_times(START_TIMES_PATH)

print("START_TIMES_PATH =", START_TIMES_PATH)
print("Loaded sessions:")
for s in session_infos:
    print(s.subject_id, s.condition, s.start_time)

print("Unique subjects in session_infos:")
print(sorted(set(s.subject_id for s in session_infos)))

# =========================
# DATASET BUILDING
# =========================

def build_signal_dataset(
    session_infos: List[SessionInfo],
    raw_root: Path,
    signal_group: str,
    signal_name: str,
    window_minutes: int,
) -> pd.DataFrame:
    collected = []

    for session in session_infos:
        session_dir = raw_root / session.subject_id / session.condition

        print(f"\nProcessing subject={session.subject_id}, condition={session.condition}, signal={signal_name}")
        print("session_dir:", session_dir)

        if not session_dir.exists():
            print(f"WARNING: missing session dir {session_dir}")
            continue

        try:
            file_path = find_signal_file(session_dir, signal_group, signal_name)
            print("file_path:", file_path)

            raw_signal = read_signal_csv(file_path, signal_group)
            print("n_raw:", len(raw_signal))

            if not raw_signal.empty and "timestamp" in raw_signal.columns:
                print("data min:", raw_signal["timestamp"].min())
                print("data max:", raw_signal["timestamp"].max())

            window = slice_session_window(
                raw_signal,
                session.start_time,
                window_minutes,
                signal_group
            )
            print("n_window:", len(window))

        except Exception as e:
            print(f"WARNING: {e}")
            continue

        print(
            f'DEBUG | signal={signal_name} | subject={session.subject_id} | '
            f'condition={session.condition} | n_raw={len(raw_signal)} | n_window={len(window)}'
        )

        if window.empty:
            continue

        window = window.copy()
        window['subject_id'] = session.subject_id
        window['condition'] = session.condition
        window['signal_group'] = signal_group
        window['signal_name'] = signal_name

        for col in OUTPUT_COLUMNS:
            if col not in window.columns:
                window[col] = pd.NA

        collected.append(window[OUTPUT_COLUMNS])

    if len(collected) == 0:
        return pd.DataFrame(columns=OUTPUT_COLUMNS)

    dataset = pd.concat(collected, ignore_index=True)
    return dataset.sort_values(
        ['condition', 'subject_id', 'timestamp']
    ).reset_index(drop=True)
# =========================
# MAIN
# =========================

def main() -> None:
    session_infos = load_start_times(START_TIMES_PATH)

    print('Loaded sessions:')
    for s in session_infos:
        print(f'  {s.subject_id} | {s.condition} | {s.start_time}')

    PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

    summary_rows = []

    for signal_group, signal_names in SIGNAL_GROUPS.items():
        for signal_name in signal_names:
            print(f"\n=== BUILDING DATASET FOR {signal_name} ===")

            dataset = build_signal_dataset(
                session_infos=session_infos,
                raw_root=RAW_ROOT,
                signal_group=signal_group,
                signal_name=signal_name,
                window_minutes=WINDOW_MINUTES,
            )

            print("Subjects included in dataset:")
            if dataset.empty:
                print("  NONE")
            else:
                print(dataset['subject_id'].unique())
                print(dataset.groupby(['subject_id', 'condition']).size())

            out_path = PROCESSED_ROOT / signal_output_name(signal_name)
            save_csv(dataset, out_path)

            print(f'SAVED {out_path} | rows={len(dataset)}')

            summary_rows.append({
                'signal_group': signal_group,
                'signal_name': signal_name,
                'output_file': str(out_path),
                'n_rows': len(dataset),
                'n_subject_condition_pairs': (
                    dataset[['subject_id', 'condition']].drop_duplicates().shape[0]
                    if not dataset.empty else 0
                ),
                'min_timestamp': dataset['timestamp'].min() if not dataset.empty else pd.NaT,
                'max_timestamp': dataset['timestamp'].max() if not dataset.empty else pd.NaT,
            })

    summary = pd.DataFrame(summary_rows)
    save_csv(summary, PROCESSED_ROOT / 'dataset_build_summary.csv')

    print(f'\nDone. Aggregated signal datasets are in:\n{PROCESSED_ROOT}')

if __name__ == "__main__":
    main()

START_TIMES_PATH = /home/eleonorab/repository/convective_cooling/data/raw/session_start_times.csv
Loaded sessions:
tester_01 no_wind 2026-04-23 10:02:37
tester_01 wind_low 2026-04-27 09:52:08
tester_02 wind_high 2026-04-29 14:50:50
Unique subjects in session_infos:
['tester_01', 'tester_02']
Loaded sessions:
  tester_01 | no_wind | 2026-04-23 10:02:37
  tester_01 | wind_low | 2026-04-27 09:52:08
  tester_02 | wind_high | 2026-04-29 14:50:50

=== BUILDING DATASET FOR T NECK ===

Processing subject=tester_01, condition=no_wind, signal=T NECK
session_dir: /home/eleonorab/repository/convective_cooling/data/raw/tester_01/no_wind
file_path: /home/eleonorab/repository/convective_cooling/data/raw/tester_01/no_wind/T/T NECK.csv
n_raw: 907
data min: 2026-04-23 09:21:31
data max: 2026-04-23 11:52:31
n_window: 360
DEBUG | signal=T NECK | subject=tester_01 | condition=no_wind | n_raw=907 | n_window=360

Processing subject=tester_01, condition=wind_low, signal=T NECK
session_dir: /home/eleonorab/rep

In [6]:
import pandas as pd

df = pd.read_csv("/home/eleonorab/repository/convective_cooling/data/processed/aggregated_by_signal/t_abdomen.csv")
print(df["subject_id"].unique())
print(df.groupby(["subject_id", "condition"]).size())

['tester_01' 'tester_02']
subject_id  condition
tester_01   no_wind      360
            wind_low     360
tester_02   wind_high    360
dtype: int64
